# 인공지능응용 · Week 14 · 문장을 Tensor로 바꾸어 감정 분류하기

**본인 사본을 만들어 실행·수정·기록하세요.**

- 이름: **⟦여기에 직접 입력⟧**
- 학번: **⟦여기에 직접 입력⟧**

[자습 자료](https://chorok-daddy.github.io/courses/ai-applications/text-classification/index.html) · [실습 안내](https://chorok-daddy.github.io/courses/ai-applications/text-classification/assignment.html)

## 시작 전 · 셀 실행과 작성 방법
Code Cell 왼쪽 ▶ 또는 Shift+Enter로 실행하세요. 위에서부터 진행하며 앞 셀의 변수를 사용합니다. Text Cell은 더블클릭해 **⟦직접 입력⟧** 부분을 바꾸고 Shift+Enter로 표시합니다. 기준 예제는 실행해서 이해하고, **직접 작성** 셀에 본인 코드를 작성하세요. 값을 바꾸면 해당 셀과 뒤의 관련 셀을 다시 실행합니다.

**60분 진행:** 예상한 결과를 기록하고 실행한 뒤, 조건을 바꾸어 비교하세요. 마지막에는 새 세션에서 다시 실행하고 작성한 설명과 파일을 확인하세요. 시간이 남으면 마지막 선택 실습을 수행하세요.

Colab에 필요한 라이브러리가 없으면 새 런타임에서 환경을 확인하고 조교에게 문의하세요. CPU로 기준 실습을 실행할 수 있습니다. 외부 다운로드가 필요한 실습은 해당 셀에 표시합니다.

## 1. 토큰과 vocabulary

문장을 Token과 Index로 바꾸고, Embedding 벡터를 평균 내어 문장 전체의 감정을 분류합니다. 이번 실습은 전처리 결과와 문장 벡터를 직접 확인하기 위해 짧은 영어 Review와 학습 가능한 Mean Embedding 모델을 사용합니다. 강의의 IMDB 예제처럼 문장 전체에 Label을 붙이되, 여기서는 세 감정으로 분류합니다. Mean Pooling은 단어 순서를 생략한 단순한 모델이며 RNN을 반드시 대체해야 해서 사용하는 것은 아닙니다. TorchText 설치, Hugging Face 로그인, 외부 모델 다운로드는 필요하지 않습니다. PAD=0은 빈자리, UNK=1은 모르는 단어입니다. screen size okay처럼 짧은 문장에 PAD가 붙는지 확인하고, 아래 비교 셀에서도 다른 길이의 문장을 사용합니다.

### A. 기준 예제 · 먼저 읽고 실행

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(42)
torch.set_num_threads(2)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
texts=['great battery bright screen','bad camera slow app','screen size okay','love fast performance','broken charger bad battery','average design normal speed','great sound love display','slow response hate keyboard','okay price average camera','bright screen fast app','bad screen okay battery','normal package average manual']
y=torch.tensor([2,0,1,2,0,1,2,0,1,2,0,1]);names=['negative','neutral','positive']
words=sorted(set(' '.join(texts).split()));vocab={'<PAD>':0,'<UNK>':1,**{w:i+2 for i,w in enumerate(words)}}
def encode(text_list):
    rows=[[vocab.get(w,1) for w in text.lower().split()] or [1] for text in text_list]
    length=max(map(len,rows));return torch.tensor([r+[0]*(length-len(r)) for r in rows])
X=encode(texts);print(texts[:3],X[:3],X.shape)


# 길이가 다를 때 PAD가 들어가는 위치를 확인합니다.
probe_texts = ['great battery', 'great battery bright screen', 'excellent battery']
probe_ids = encode(probe_texts)
for text, row in zip(probe_texts, probe_ids):
    print('text:', text, '| tokens:', text.lower().split(), '| ids:', row.tolist())
assert (probe_ids[0] == 0).sum().item() == 2
assert probe_ids[2, 0].item() == vocab['<UNK>']


### B. 직접 수행

길이가 다른 두 영어 문장을 Token → Index → PAD 순서로 출력하세요. 학습 사전에 없는 단어 하나를 넣고 UNK 위치를 확인하세요. 실제 단어를 뜻하는 Index와 PAD·UNK를 구별해 적으세요.

예상은 정확한 수치 대신 shape나 증가·감소 방향으로 적어도 됩니다. 설명은 아래 Text Cell에 기록하세요.


In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 확인하거나 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 2. 마스킹한 평균과 학습

12·13주차는 문자마다 다음 문자를 예측했지만 이번에는 문장당 감정 하나를 예측합니다. Embedding (N,S,16)에서 PAD를 뺀 평균 (N,16)을 만든 뒤 Linear로 (N,3) Logit을 만듭니다. Embedding과 Linear는 함께 학습되며, 이 모델에는 RNN/LSTM이 없습니다.

### A. 기준 예제 · 먼저 읽고 실행

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self):
        super().__init__();self.embedding=nn.Embedding(len(vocab),16,padding_idx=0);self.fc=nn.Linear(16,3)
    def forward(self,ids):
        mask=(ids!=0).unsqueeze(-1);embedded=self.embedding(ids)
        pooled=(embedded*mask).sum(1)/mask.sum(1).clamp_min(1)
        return self.fc(pooled)
model=TextClassifier();opt=torch.optim.Adam(model.parameters(),lr=.02);losses=[];accuracies=[]
for epoch in range(200):
    logits=model(X);loss=F.cross_entropy(logits,y)
    opt.zero_grad();loss.backward();opt.step()
    losses.append(loss.item());accuracies.append((logits.argmax(1)==y).float().mean().item())
print('Training accuracy:',accuracies[-1]);print('embedding:',model.embedding(X).shape)
fig,axes=plt.subplots(1,2,figsize=(9,3));axes[0].plot(losses);axes[1].plot(accuracies)
for ax,yl in zip(axes,['Loss','Accuracy']):ax.set(xlabel='Epoch',ylabel=yl);ax.grid()
plt.show()


# 같은 문장을 혼자 처리하거나 긴 문장과 묶어도, 올바른 평균은 같습니다.
with torch.no_grad():
    single_ids = encode(['great battery'])
    batch_ids = encode(['great battery', 'great battery bright screen'])
    single_vector = model.embedding(single_ids).mean(1)[0]
    batch_embedded = model.embedding(batch_ids)
    mask = (batch_ids != 0).unsqueeze(-1)
    masked_vectors = (batch_embedded * mask).sum(1) / mask.sum(1).clamp_min(1)
    plain_vectors = batch_embedded.mean(1)
    print('PAD 포함 평균과 차이:', (single_vector - plain_vectors[0]).abs().max().item())
    print('PAD 제외 평균과 차이:', (single_vector - masked_vectors[0]).abs().max().item())
    assert torch.allclose(single_vector, masked_vectors[0], atol=1e-6)
    order_logits = model(encode(['great battery', 'battery great']))
    print('단어 순서를 바꾼 두 문장의 Logit:', order_logits)
    assert torch.allclose(order_logits[0], order_logits[1], atol=1e-6)


### B. 직접 수행

같은 문장을 혼자 처리할 때와 긴 문장과 묶을 때 Mask 적용 전후 평균을 비교하세요. 다른 문장 쌍으로 한 번 더 확인하고, 단어 순서를 바꾼 두 문장의 Logit도 비교하세요. 차이가 0에 가까운 이유를 설명하세요.

예상은 정확한 수치 대신 shape나 증가·감소 방향으로 적어도 됩니다. 설명은 아래 Text Cell에 기록하세요.


In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 확인하거나 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 3. 새 문장 직접 입력

아래 my_reviews의 빈 문자열 세 개를 영어 문장으로 바꾸세요. 처음에는 texts에 나온 단어를 사용하고, 다음에는 모르는 단어 하나를 추가합니다. 의도한 감정과 모델 예측이 다르더라도 결과와 이유를 기록하세요. 미작성 상태에서는 예측을 건너뜁니다.

### A. 기준 예제 · 먼저 읽고 실행

In [ ]:
my_reviews=['','',''] # 직접 입력: 긍정 1개, 중립 1개, 부정 1개
if all(s.strip() for s in my_reviews):
    model.eval()
    with torch.no_grad():prob=model(encode(my_reviews)).softmax(1)
    for text,p in zip(my_reviews,prob):print(text,'=>',names[p.argmax().item()],p.tolist())
else:
    print('my_reviews의 빈 문자열 3개를 직접 작성하고 이 셀을 다시 실행하세요.')
print('unknown token example:',encode(['excellent battery']))



### B. 직접 수행

학습 사전의 단어로 긍정·중립·부정 영어 문장을 하나씩 my_reviews에 작성하세요. 의도한 감정·예측·Token을 기록하고, 모르는 단어나 문장부호를 하나 넣어 UNK 처리를 확인하세요. 예측이 의도와 달라도 원인을 설명하면 됩니다.

예상은 정확한 수치 대신 shape나 증가·감소 방향으로 적어도 됩니다. 설명은 아래 Text Cell에 기록하세요.


In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 확인하거나 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 반복 숙달 · 실행 전에 판단하기

앞의 직접 수행에서 비교한 조건 세 가지를 이 표에 정리하세요. 이미 수행한 실험을 다시 추가할 필요는 없습니다. 비교가 부족하면 한 조건만 바꾸어 보충하세요.

| 변경 조건 | 예상 shape/수치/결과 | 실제 결과 | 오류가 있었다면 원인 |
|---|---|---|---|
| ⟦직접 입력 1⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |
| ⟦직접 입력 2⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |
| ⟦직접 입력 3⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |

In [ ]:
# ✍ 반복 실험 코드


## 선택 실습 · 오류의 원인을 찾아 코드 고치기

앞 코드의 축·dtype·입력 크기·모델 설정 중 하나를 일부러 바꿔 예상과 달라지는 사례를 만드세요. 오류를 그대로 남기지 말고, 어떤 입력 조건이나 연산 규칙이 맞지 않았는지 설명한 뒤 수정된 코드로 실행하세요. 인증·설치 설정을 바꾸는 실험은 하지 않습니다.

## 저장 전 확인

1. 직접 작성란을 채우고 필요한 출력·그래프를 남겼는지 확인합니다.
2. 새 런타임에서 위에서부터 실행해 숨은 변수 의존성을 확인합니다.
3. 수정한 파일을 본인 Drive에 저장하고 `.ipynb`로 내려받습니다.
4. 제출 파일명·기한은 블로그의 이번 주 실습 안내와 최신 KLAS 공지를 따릅니다.